### Get data from hoogspanningsnet.com

In [101]:
import geopandas as gpd
from shapely.geometry import Point
import json
from collections import defaultdict
spanningscutoff = 100
use_tennet_stations = True
cutoff = 0.00001 # max distance between connections
# max_distance_line_station = 0.0005 # not too large since Borselle becomes ZKL
max_distance_line_station = 0.004 # not too large since Borselle becomes ZKL
max_distance = 0.01 # max distance between stations

accept_multiple_lines = False # for drawing
scale = 10 # the amount to scale with (higher --> more scaling down)
cluster_stations = True
spanning_kleuren = {
    380: 'red',
    150: 'blue',
    220: 'forestgreen',
    110: 'black'
}  

In [102]:
def inspect_netschakel(netschaekl):
    for n in netschakels[netschaekl]:
        print(n, id2conn[n]['properties']['from_id'], id2conn[n]['properties']['to_id'])
    
# for n in netschakels:
#     if n and 'BGM110-BGM010' in n: print(n, inspect_netschakel(n), '\n')

In [103]:
import geopandas as gpd
from shapely.geometry import Point
import json

# Laad de Nederland shapefile (EPSG:3035)

nl = gpd.read_file("netherlands_country_boundary/netherlands_Netherlands_Country_Boundary.shp")
country_shape = nl.unary_union  # merge all polygons once

def punt_in_nederland(lon, lat, boundary=country_shape):
    punt = Point(lon, lat)
    return boundary.contains(punt)

# Voorbeeld gebruik
print(punt_in_nederland(4.895, 52.370))  # Amsterdam, True|
print(punt_in_nederland(6.14, 49.78))    # Buiten Nederland, False
print(punt_in_nederland(7.404059, 52.2884293))    # Buiten Nederland, False
print(punt_in_nederland(*[6.926667, 51.498889]))    # Buiten Nederland, False
print(punt_in_nederland(*[6.6308981, 51.0604542]))    # Buiten Nederland, False
print(punt_in_nederland(*[7.03359, 52.202448]))    # Buiten Nederland, False
print(punt_in_nederland(*[7.0325,52.2016]))    # Buiten Nederland, False

# Compute centroids of Netherlands
x = float(nl.geometry.centroid.x.iloc[0])
y = float(nl.geometry.centroid.y.iloc[0])

True
False
False
False
False
False
False


C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_20216\1692540911.py:8: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  country_shape = nl.unary_union  # merge all polygons once
C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_20216\1692540911.py:24: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  x = float(nl.geometry.centroid.x.iloc[0])
C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_20216\1692540911.py:25: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  y = float(nl.geometry.centroid.y.iloc[0])


In [104]:
import requests
import networkx as nx
import matplotlib.pyplot as plt
from math import sqrt
from urllib.parse import urlparse, parse_qs

url = 'https://webkaart.hoogspanningsnet.com/layerdata.php?type=trms&zoom=8&bbox=4.086914062500001%2C51.23784668914442%2C17.869262695312504%2C53.68044193408406'

# URL parsen
parsed = urlparse(url)
query = parse_qs(parsed.query)
bbox = "3.315,50.775,7.226,53.576"

# Stations ophalen (stic)
def get_url(t, zoom=14):
    url = f"https://webkaart.hoogspanningsnet.com/layerdata.php?type={t}&zoom={zoom}&bbox={bbox}"
    return requests.get(url).json()

def getconn(num):
    if use_tennet_stations:
        return next(f for f in verbindingen['features'] if f['properties']['ID'] == str(num))
    else:
        return next(f for f in verbindingen['features'] if f['properties']['ID'] == 'v'+str(num))

def findstation(naam):
    return next(s for s in hoogspanning_stations['features'] if s['properties']['Naam']==naam)
hoogspanning_stations = get_url('stic')
hoogspanning_verbindingen = get_url('trvb')
hoogspanning_polygons = get_url('sttr', zoom=14) # werkt alleen bij zoom = 14 of gedetailleerder
knooppunten = get_url('trkp', zoom=14) 
masten = get_url('trms', zoom=14) 

for var in [hoogspanning_stations, hoogspanning_verbindingen, hoogspanning_polygons, knooppunten, masten]:
    print(len(var['features']))


2033
7596
2979
193
30534


### Remove stations and lines that are not relevant

In [105]:


hoogspanning_stations['features'] = [f for f in hoogspanning_stations['features'] if punt_in_nederland(*f['geometry']['coordinates']) and f['properties']['Spanning']>= spanningscutoff]
hoogspanning_polygons['features'] = [f for f in hoogspanning_polygons['features'] if punt_in_nederland(*f['geometry']['coordinates'][0][0]) and f['properties']['Spanning']>= spanningscutoff]
hoogspanning_verbindingen['features'] = [f for f in hoogspanning_verbindingen['features'] if (punt_in_nederland(*f['geometry']['coordinates'][0]) or punt_in_nederland(*f['geometry']['coordinates'][-1])) and f['properties']['Spanning']>= spanningscutoff]
print(len(hoogspanning_stations['features']), len(hoogspanning_polygons['features']), len(hoogspanning_verbindingen['features']))

458 593 1446


### Alternatively load TenneT Geojsons

In [106]:
import json
from itertools import chain
with open("tennet/Hoogspanning_station.geojson", "r", encoding="utf-8") as f:
    geo = json.load(f)
with open("tennet/Opstijgpunt.geojson", "r", encoding="utf-8") as f:
    geo2 = json.load(f)
  
tennet = {}
tennet['features'] = []
for feature in chain(geo['features']): #, geo2['features']):
    props = feature["properties"]
    if "SE_FLD55_SPANNINGSNIVEAU" in props:
        props["Spanning"] = props.pop("SE_FLD55_SPANNINGSNIVEAU")
    if "SE_FLD24_OBJECTOMSCHRIJVING" in props:
        props["Naam"] = props.pop("SE_FLD24_OBJECTOMSCHRIJVING")
        props["Osp"] = False
    elif 'SE_FLD13_OBJECTID' in props: # opstijgpunt
        props["Naam"] = props.pop("SE_FLD13_OBJECTID")
        props["Osp"] = True
        
        
    if feature["geometry"] is not None and props['Naam'] is not None:
        idx = props["Naam"].rfind(' ')
        # print(props["Naam"])
        if idx > 8 and props["Naam"].startswith('Station'): props["Naam"] = props["Naam"][8:idx]
        if props["Naam"][-1].isdigit():
            idx = props["Naam"].rfind(' ')
            props["Naam"] = props["Naam"][:idx]
        if props["Spanning"] > spanningscutoff or props['Naam'] == 'Oudehaske':
            tennet['features'].append(feature)
            if 'coordinates' not in feature['geometry']:
                print(feature)
with open("tennet/Hoogspanning_kabel_(ondergronds).geojson", "r", encoding="utf-8") as f:
    geo = json.load(f)
with open("tennet/Hoogspanning_leiding_(bovengronds).geojson", "r", encoding="utf-8") as f:
    geo2 = json.load(f)
tennet_verbindingen = {
    "type": "FeatureCollection",
    "features": geo["features"] + geo2["features"]
}
for feature in tennet_verbindingen['features']:
    feature['properties']['ID'] = feature['properties']['SE_FLD32_OBJECTID'] if 'SE_FLD32_OBJECTID' in feature['properties'] else feature['properties']['SE_FLD33_OBJECTID']
    feature['properties']['Spanning'] = feature['properties']['SE_FLD38_SPANNINGSNIVEAU'] if 'SE_FLD38_SPANNINGSNIVEAU' in feature['properties'] else feature['properties']['SE_FLD39_SPANNINGSNIVEAU']
    feature['properties']['Netschakel'] = feature['properties']['SE_FLD27_NETSCHAKELID'] if 'SE_FLD27_NETSCHAKELID' in feature['properties'] else feature['properties']['SE_FLD28_NETSCHAKELID']
    
tennet_verbindingen['features'] = [f for f in tennet_verbindingen['features'] if f['geometry']['type'] == 'LineString' and feature['properties']['Spanning'] > spanningscutoff]

In [107]:
for s in tennet['features']:
    if 'Oude' in s['properties']['Naam']:
        print(s['properties'])

{'SE_FLD1_BEDRIJFSSTATUS': 'In bedrijf', 'SE_FLD4_BOUWJAAR': 1262304000000, 'ESRI_OID': 25, 'SE_FLD23_OBJECTID': 'EOS380', 'Shape__Area': 36061.02653503418, 'Shape__Length': 751.9155389363436, 'Spanning': 380, 'Naam': 'Eemshaven Oudeschip', 'Osp': False}
{'SE_FLD1_BEDRIJFSSTATUS': 'In bedrijf', 'SE_FLD4_BOUWJAAR': 599616000000, 'ESRI_OID': 126, 'SE_FLD23_OBJECTID': 'ODL150', 'Shape__Area': 5023.072624206543, 'Shape__Length': 282.0907012690734, 'Spanning': 150, 'Naam': 'Oudeland', 'Osp': False}
{'SE_FLD1_BEDRIJFSSTATUS': 'In bedrijf', 'SE_FLD4_BOUWJAAR': 852076800000, 'ESRI_OID': 127, 'SE_FLD23_OBJECTID': 'ODR150', 'Shape__Area': 6395.154697418213, 'Shape__Length': 326.62267415540947, 'Spanning': 150, 'Naam': 'Oudenrijn', 'Osp': False}
{'SE_FLD1_BEDRIJFSSTATUS': 'In bedrijf', 'SE_FLD4_BOUWJAAR': 189302400000, 'ESRI_OID': 324, 'SE_FLD23_OBJECTID': 'OHK220', 'Shape__Area': 28976.01528930664, 'Shape__Length': 807.4463518609955, 'Spanning': 220, 'Naam': 'Oudehaske', 'Osp': False}
{'SE_FLD1_

### Determine stationsnamen of polygons

In [108]:
from math import sqrt, dist
from itertools import combinations
from copy import deepcopy

if use_tennet_stations:
    stations = deepcopy(tennet)
    verbindingen = deepcopy(tennet_verbindingen)
    verbindingen['features'] = [v for v in verbindingen['features'] if v['properties']['Spanning'] > spanningscutoff]
    station_polygons = deepcopy(tennet)
else:
    stations = deepcopy(hoogspanning_stations)
    verbindingen = deepcopy(hoogspanning_verbindingen)
    station_polygons = hoogspanning_polygons
    
id2conn = {f['properties']['ID']: f for f in verbindingen['features']}
not_found = {f['properties']['Naam'] for f in stations['features']}

if not use_tennet_stations:
    cnt = 1
    matched = 0
    for poly in station_polygons['features']:
        coords = poly['geometry']['coordinates'][0]
        avg_x = sum(p[0] for p in coords)/len(coords)
        avg_y = sum(p[1] for p in coords)/len(coords)
        poly_point = (avg_x, avg_y)
        
        # Vind het dichtstbijzijnde station
        min_dist = float('inf')
        closest_station_name = None
        for station in stations['features']:
            if use_tennet_stations:
                station_point = station['geometry']['coordinates'][0][0]
            else:
                station_point = station['geometry']['coordinates']
                
            d = dist(poly_point, station_point)
            if d < min_dist:
                min_dist = d
                closest_station = station
        
        if min_dist <= max_distance_line_station:
            poly['properties']['Naam'] = closest_station['properties']['Naam']
            not_found.discard(closest_station['properties']['Naam'])
            matched +=1 
        else:
            cnt +=1
    print(matched)
    print(len(not_found))
    not_found # Nuon Magnumcentrale is wel even interessant om in de gaten te houden



In [109]:
id2conn = {f['properties']['ID']: f for f in verbindingen['features']}
netschakels = defaultdict(list)
if use_tennet_stations:
    for v in verbindingen['features']:
        netschakels[v['properties']['Netschakel']].append(v['properties']['ID'])
        
coortonet = defaultdict(list)
for n in netschakels:
    for conn in netschakels[n]:
        c = id2conn[conn]['geometry']['coordinates'][0]
        coortonet[tuple(c)].append((n,conn))
        c = id2conn[conn]['geometry']['coordinates'][-1] # only the extremeties
        coortonet[tuple(c)].append((n,conn))
print(len(coortonet))
coortonet = {k:v for k,v in coortonet.items() if len(v)>1}
print(len(coortonet))
coortonet = {k:v for k,v in coortonet.items() if len(v)>2}
print(len(coortonet))

95238
88112
271


In [110]:
stations['features'].append(stations['features'][-1])
stations['features'][-1]['geometry']['coordinates'] = [[[5.90973482453061, 51.6376804275618],[5.90978916894353, 51.6376950772005],[5.90984351340179, 51.6377097277127]]]
stations['features'][-1]['properties']['Naam'] = 'Boxmeer'
stations['features'][-1]['properties']['Spanning'] = 380

### Determine start and endlocation of connections

In [111]:
from functools import cache
import math

def point_in_polygon_strict(point, polygon):
    """
    Bepaal of een punt binnen een polygon ligt met ray-casting.
    point: (x, y)
    polygon: lijst van (x, y) tuples
    """
    x, y = point
    inside = False
    n = len(polygon)
    
    p1x, p1y = polygon[0]
    for i in range(n + 1):
        p2x, p2y = polygon[i % n]
        if y > min(p1y, p2y):
            if y <= max(p1y, p2y):
                if x <= max(p1x, p2x):
                    if p1y != p2y:
                        xinters = (y - p1y)*(p2x - p1x)/(p2y - p1y) + p1x
                    if p1x == p2x or x <= xinters:
                        inside = not inside
        p1x, p1y = p2x, p2y
    return inside

def point_line_distance(px, py, x1, y1, x2, y2):
    """Afstand van punt (px,py) tot lijnstuk (x1,y1)-(x2,y2)."""
    dx, dy = x2 - x1, y2 - y1
    if dx == dy == 0:
        return math.hypot(px - x1, py - y1)  # lijnstuk is een punt
    t = max(0, min(1, ((px - x1) * dx + (py - y1) * dy) / (dx*dx + dy*dy)))
    nx, ny = x1 + t*dx, y1 + t*dy
    return math.hypot(px - nx, py - ny)


def point_in_polygon_less_strict(point, polygon, epsilon=0.001):
    """
    Bepaal of een punt binnen een polygon ligt met ray-casting,
    of binnen 'epsilon' afstand van de rand.
    """
    x, y = point
    inside = False
    n = len(polygon)
    
    p1x, p1y = polygon[0]
    for i in range(n + 1):
        p2x, p2y = polygon[i % n]
        if y > min(p1y, p2y):
            if y <= max(p1y, p2y):
                if x <= max(p1x, p2x):
                    if p1y != p2y:
                        xinters = (y - p1y) * (p2x - p1x) / (p2y - p1y) + p1x
                    if p1x == p2x or x <= xinters:
                        inside = not inside
        # check afstand tot rand
        if point_line_distance(x, y, p1x, p1y, p2x, p2y) <= epsilon:
            return True
        p1x, p1y = p2x, p2y

# Eerst bounding boxes van alle polygons berekenen
# Precompute bounding boxes and polygon lengths once
polygon_bboxes = []
for feature in station_polygons['features']:
    if 'Naam' in feature['properties']:
        coords = feature['geometry']['coordinates'][0]
        xs = [p[0] for p in coords]
        ys = [p[1] for p in coords]
        polygon_bboxes.append({
            'ID': feature['properties']['Naam'],
            'min_x': min(xs),
            'max_x': max(xs),
            'min_y': min(ys),
            'max_y': max(ys),
            'polygon': coords,
            'spanning': feature['properties']['Spanning'],
            'n': len(coords)  # precompute once
        })

    
@cache
def find_polygon(x, y, epsilon=0.001):
    
    """Return the polygon ID containing the point, strict first, then epsilon margin"""
    # if use_tennet_stations: epsilon=0.000
    for bbox in polygon_bboxes:
        # Quick bounding box check
        if not (bbox['min_x'] <= x <= bbox['max_x'] and bbox['min_y'] <= y <= bbox['max_y']):
            continue  # skip polygons that can't contain the point
        # Strict point-in-polygon
        inside = False
        n = bbox['n']
        p1x, p1y = bbox['polygon'][0]
        for i in range(n + 1):
            p2x, p2y = bbox['polygon'][i % n]
            if y > min(p1y, p2y):
                if y <= max(p1y, p2y):
                    if x <= max(p1x, p2x):
                        if p1y != p2y:
                            xinters = (y - p1y)*(p2x - p1x)/(p2y - p1y) + p1x
                        if p1x == p2x or x <= xinters:
                            inside = not inside
            # less strict: check epsilon distance to edge
            dx, dy = p2x - p1x, p2y - p1y
            if dx == dy == 0:
                d = math.hypot(x - p1x, y - p1y)
            else:
                t = max(0, min(1, ((x - p1x) * dx + (y - p1y) * dy) / (dx*dx + dy*dy)))
                nx, ny = p1x + t*dx, p1y + t*dy
                d = math.hypot(x - nx, y - ny)
            if d <= epsilon:
                # print(d)
                return bbox['ID']
            p1x, p1y = p2x, p2y

        if inside:
            return bbox['ID']

    return False


# Graaf opbouwen
G = nx.MultiGraph()

# Controleer elk punt van elke verbinding
for feature in verbindingen['features']:
# for feature in [getconn(606969)]:

    conn_list = []
    # print(feature['properties']['ID'])
    verbinding_id = feature['properties']['ID']
    


    for point in feature['geometry']['coordinates']:
        if (res:=find_polygon(*point)):
            conn_list.append(res)
        
    if len(conn_list) > 1:
        color = spanning_kleuren.get(feature['properties']['Spanning'], 'grey')
        for a,b in combinations(set(conn_list), 2):
            G.add_edge(a, b, color=color)
        
    feature['properties']['from_id'] = find_polygon(*feature['geometry']['coordinates'][0])
    feature['properties']['to_id'] = find_polygon(*feature['geometry']['coordinates'][-1])
    feature['properties']['connections'] = set(conn_list)
    if len(set(conn_list)) > 2:
        pass
        # print(feature['properties']['ID'], feature['properties']['Spanning'],conn_list)

In [112]:
for p in getconn(606960)['geometry']['coordinates']:
    print(find_polygon(*p))

Oudehaske
False


In [113]:
for p in getconn(606960)['geometry']['coordinates']:
    print(find_polygon(*p))

Oudehaske
False


In [114]:
for p in getconn(496052)['geometry']['coordinates']:
    print(find_polygon(*p))

False
Lelystad


### at this point, not all lines and cables have a start and endstation, lets fix that

In [115]:
from itertools import product
from aocutils.special import UnionFind
from itertools import product

def parallellines(vid1, vid2):
    start1 = vid1['geometry']['coordinates'][0]
    end1 = vid1['geometry']['coordinates'][-1]
    start2 = vid2['geometry']['coordinates'][0]
    end2 = vid2['geometry']['coordinates'][-1]
    options = product([start1, end1], [start2, end2])
    return sum(dist(*o) < (cutoff * 1) for o in options) >= 2

In [132]:
from copy import deepcopy
def process_connections(results):
    found = []
    unique_connections = []
    for option in results:
        start = id2conn[option[0]]['properties']['from_id'] if id2conn[option[0]]['properties']['from_id'] else id2conn[option[0]]['properties']['to_id'] 
        end = id2conn[option[-1]]['properties']['from_id'] if id2conn[option[-1]]['properties']['from_id'] else id2conn[option[-1]]['properties']['to_id']
        if sorted([start, end]) not in found:
            found.append(sorted([start, end]))
            unique_connections.append(deepcopy(id2conn[option[0]]))
            if not id2conn[option[0]]['properties']['from_id']: 
                unique_connections[-1]['geometry']['coordinates'] = unique_connections[-1]['geometry']['coordinates'][::-1]
            unique_connections[-1]['properties']['from_id'] = start
            unique_connections[-1]['properties']['to_id'] = end
            for lineid in option[1:]:
                if id2conn[lineid]['geometry']['coordinates'][0] == unique_connections[-1]['geometry']['coordinates'][-1]:
                    unique_connections[-1]['geometry']['coordinates'] += id2conn[lineid]['geometry']['coordinates'][1:]
                else:
                    unique_connections[-1]['geometry']['coordinates'] += id2conn[lineid]['geometry']['coordinates'][::-1][1:]
    print(len(results), len(unique_connections), found, [len(u['geometry']['coordinates']) for u in unique_connections] )
    return unique_connections
            

In [133]:
def dfs(cur, path, goals):
    results = []
    for neighbor in neigh[cur]:
        if neighbor not in path:
            if neighbor in goals:
                return [path + [neighbor]]
            else:
                path.append(neighbor)
                for r in dfs(neighbor, path, goals):
                    results.append(r)
                assert path.pop() == neighbor
    # if '714529' in path: print(results, path)
    return results

            
        

  
from operator import xor
from copy import deepcopy
# make a dict
def make_coor2id(netschakel):
    coor2id = defaultdict(list)
    for lineid in netschakels[netschakel]:
        coor2id[tuple(id2conn[lineid]['geometry']['coordinates'][0])].append(lineid)
        coor2id[tuple(id2conn[lineid]['geometry']['coordinates'][-1])].append(lineid)
    return coor2id

def make_neighbors(coor2id):
    conn = defaultdict(set)
    for connected in coor2id.values():
        for c1 in connected:
            for c2 in connected:
            
                if c1 != c2:
                    conn[c1].add(c2)
    return conn
    




finalset = {}
finalset['features'] = []
for netschakel in netschakels:
    results = []
    coor2id = make_coor2id(netschakel) # per coordinate al the lines
    neigh = make_neighbors(coor2id) # per line all it's neighbors (on both sides)
    # get starting lines
    starting = []
    for lineid in netschakels[netschakel]:
        line = id2conn[lineid]
        start = line['properties']['from_id']
        end = line['properties']['to_id']    
        if xor(start is False, end is False): 
            starting.append(lineid)
    for startinglineid in starting:
        results += dfs(startinglineid, [startinglineid], starting)
    if len(results) == 0 and len(coor2id) > 50 and len(starting) >= 6:
        print(netschakel, len(results), len(neigh), len(coor2id),list(coor2id.keys())[0], starting, [id2conn[s]['properties']['from_id'] if id2conn[s]['properties']['from_id'] else id2conn[s]['properties']['to_id'] for s in starting])
        if '712417' in neigh: print(neigh['712417'])
        
    finalset['features'] += process_connections(results)
         
        
# TL-ZBM150 W 0 192 198 (5.44550381702264, 51.8982611957883)
# RSD-RSB-WDT150 W 0 216 225 (4.40250242763672, 51.5076192319845) ['715165', '715175', '818130', '715169', '714490', '818133', '714494', '818128', '714512']

0 0 [] []
6 1 [['North Sea Wind', 'Velsen']] [860]
0 0 [] []
0 0 [] []
0 0 [] []
6 1 [['Dinteloord', 'Roosendaal']] [3845]
0 0 [] []
0 0 [] []
0 0 [] []
0 0 [] []
0 0 [] []
0 0 [] []
0 0 [] []
0 0 [] []
0 0 [] []
0 0 [] []
0 0 [] []
6 1 [['Bleiswijk', 'Zoetermeer']] [347]
0 0 [] []
2 1 [['Enschede Vechtstraat', 'Enschede Wesselerbrink']] [507]
0 0 [] []
6 1 [['Norg', 'Zeyerveen']] [836]
6 1 [['Bleiswijk', 'Zoetermeer']] [387]
0 0 [] []
0 0 [] []
2 1 [['Hoge snelheidslijn', 'Zoetermeer']] [190]
36 3 [['Heerenveen', 'Rauwerd'], ['Heerenveen', 'Oudehaske'], ['Oudehaske', 'Rauwerd']] [309, 242, 100]
6 1 [['Ommoord', 'Rotterdam Marconistraat']] [1594]
0 0 [] []
0 0 [] []
0 0 [] []
0 0 [] []
0 0 [] []
0 0 [] []
0 0 [] []
6 1 [['Hollandse Kust Zuid Landstation', 'Maasvlakte']] [276]
0 0 [] []
0 0 [] []
0 0 [] []
6 1 [['Hollandse Kust Zuid Landstation', 'Maasvlakte']] [279]
0 0 [] []
6 1 [['Emmeloord Zuidervaart', 'Luttelgeest Kalenbergerweg']] [1433]
6 1 [['Borssele', 'Vlissingen Oost']] [155

In [ ]:
def dfs(origin, origincoor, coor2id):
    results = []
    for neighbor in coor2id[tuple(origincoor)]:
        
        if neighbor != origin:
            neighborline = id2conn[neighbor]
            start = neighborline['properties']['from_id']            
            end = neighborline['properties']['to_id']
            if start: return [[neighborline['geometry']['coordinates'], start]]
            elif end: return [[neighborline['geometry']['coordinates'], end]]
            else:
                startcoorneigh = neighborline['geometry']['coordinates'][0]
                endcoorneigh = neighborline['geometry']['coordinates'][-1]
                if startcoorneigh == origincoor:
                    currentcoordinates = neighborline['geometry']['coordinates'][1:]
                    res = dfs(neighbor, endcoorneigh, coor2id)
                else:
                    currentcoordinates = neighborline['geometry']['coordinates'][::-1][1:]
                    res = dfs(neighbor, startcoorneigh, coor2id)
            if res:
                for coordinates, endpoint in res:
                    results.append([currentcoordinates + coordinates, endpoint])
    return results
            
    
from operator import xor
from copy import deepcopy
# make a dict
def make_coor2id(netschakel):
    coor2id = defaultdict(list)
    for lineid in netschakels[netschakel]:
        for coor in id2conn[lineid]['geometry']['coordinates']:
            coor2id[tuple(coor)].append(lineid)
    return coor2id






finalset = {}
finalset['features'] = []
for netschakel in netschakels:
    print(netschakel)
    # netschakel = 'MBT-BMR-DOD380 Z'
    coor2id = make_coor2id(netschakel)
    realconnections = []
    for lineid in netschakels[netschakel]:
        line = id2conn[lineid]
        start = line['properties']['from_id']
        end = line['properties']['to_id']
        realwinningline = deepcopy(line)
        if xor(start is False, end is False): 
            if start: 
                res = dfs(lineid, line['geometry']['coordinates'][-1], coor2id)
            else:
                realwinningline['geometry']['coordinates'] = realwinningline['geometry']['coordinates'][::-1]
                realwinningline['properties']['from_id'] = end
                realwinningline['properties']['to_id'] = False
                res = dfs(lineid, line['geometry']['coordinates'][0], coor2id)
            # print('starting', start, end)
            for restcoordinates, endlocation in res:
                newrealwinningline = deepcopy(realwinningline)
                newrealwinningline['geometry']['coordinates'] += restcoordinates
                newrealwinningline['properties']['to_id'] = endlocation
                realconnections.append(newrealwinningline)
                # print(endlocation)
        else:
            # either part in the middle or fully within a station
            pass
    finalset['features'].extend(realconnections)


MDH150-MDH050 TR4 nvt
VLN150-NSW Nvt
DK110-DK010 TR102
RTW150-RTW025 TR3
DDM150-DDM050 TR3
DTO-RSD150 W
APL150-APL050 TR2 Nvt
MDH150-MDH050 TR5 nvt
YPB150-YPB25 TR5
MDH150-MDH050 TR7 nvt
MVL150-MVL066 TR4
MDH150-MDH050 TR6 nvt
APL150-APL050 TR1 Nvt
GNHK110-GNHK010 TR112
YPB150-YPB25 TR2
SNB110-SNB010 TR9001
ZBK150-ZBK020 TR202
BWK380-ZT150 TR412
WTR380-WTR150 TR411
ESDV-ESDW110 W
ZBK150-ZBK020 TR101
NO-ZYV110 W
BWK380-ZT150 TR414
RTW150-RTW025 TR2
RTW150-RTW025 TR1
ZT-HSL150 TR102
RWD-HRV-OHK110 Z
RTM-OM150 Z
ZT-HSL150 TR1
WDO150-HAS20 W
MVL150-MVL025 TR6
WDO150-HAS10 W
GLTK110-GLTK020 TR121
HZL-HZA220 Z
HZL-HZA220 W
MVL-HZL380 W
HZL-HZB220 P
DDM150-DDM050 TR1
WDC-EDR400 Nvt
MVL-HZL380 Z
HZL-HZB220 O
LTK-EMZ110 Z
BSL-VLO150 W
CST380-DDM150 TR403
BSL-VLO150 Z
MEE-VDMZ110 W
MEE-VDMZ110 Z
LTK-EMZ110 W
BN-AMV150 Z
ERP-DTH150 Z
SK110-SK010 TR101
OM150-OM025 TR2
RBB-OPD220 W
OM150-OM025 TR1
AB150-AB050 TR1
RBB-OPD220 Z
ESDM110-ESM10
BTL-AKM150 W
VW150-VW010 TR1 Nvt
MVL150-MVL066 TR5
GT150-GT

RecursionError: maximum recursion depth exceeded

: 

In [ ]:
realconnections

[{'type': 'Feature',
  'id': 61404,
  'geometry': {'type': 'LineString',
   'coordinates': [[5.91247640300767, 51.6382715074951],
    [5.90978916894353, 51.6376950772005],
    [5.90949192699319, 51.6359098109517],
    [5.90919470722789, 51.63412454273],
    [5.90862826511394, 51.6305922995276],
    [5.90805765366014, 51.627114466233],
    [5.90749739853637, 51.6237167472276],
    [5.90692865703274, 51.6202855528133],
    [5.90663680744279, 51.6185736005156],
    [5.90635384660984, 51.6168526568684],
    [5.90580561996849, 51.6135180584115],
    [5.9052178000669, 51.6099420334082],
    [5.90463008789554, 51.60636600425],
    [5.90404245435506, 51.6027899801247],
    [5.9037486819418, 51.6010020167885],
    [5.90345493148508, 51.599214051515],
    [5.90287808496819, 51.5956373343965],
    [5.90229152656859, 51.5920615544147],
    [5.90170485078636, 51.5884873813827],
    [5.90112640033347, 51.5849674066489],
    [5.90083659487715, 51.5832015913063],
    [5.90054681211988, 51.581435775181

In [ ]:
# TODO pay attention to the lines connecting 3 stations
# The lines below going nowhere are often interconnectors or offshore cables (outside of NL)

### Now we know start and end locations and need to merge connecting connections with DFS (eg a line and a cable)

In [ ]:
True * False

0

491828 491836
535449 492899
535722 493922
494410 494422
494502 494520
492677 535416
489610 489656
492897 492901
491271 491295
535214 491564
492741 492745
494186 494194
491473 535218
493264 493270
491169 491193
491171 491195
494057 494069
495135 495151
538607 493974
492508 535396
551660 494370
492528 492532
489567 489601
491798 491802
489539 489579
494188 494196
492172 492210
489767 489795
494376 494380
491710 491798
491814 492182
491594 491722
494178 494182
494077 507571
494909 494913
538615 494027
495055 495069
491263 491287
493926 535755
495153 495165
492903 492907
495214 553142
492170 492270
493258 535600
535726 493920
491165 491171
491399 491463
490963 490965
491141 491165
494941 494945
507589 494133
489922 535090
495391 495470
493930 493946
539321 495097
495121 495137
491553 535246
491197 491229
489181 489490
494149 538775
495300 495316
494458 551652
495163 495179
507528 492420
491564 491572
493117 493119
535238 491712
493129 535571
489846 489870
494915 494919
489876 489930
493416

In [ ]:
for n in ['MBT-BMR-DOD380 Z']:

defaultdict(list,
            {'MDH150-MDH050 TR4 nvt': ['1438576', '1438581', '1438593'],
             'VLN150-NSW Nvt': ['1665620',
              '1665612',
              '1665621',
              '1665615',
              '1665627',
              '1665626',
              '1665624',
              '1665625',
              '1665617',
              '1665614',
              '1665632',
              '1665619',
              '1665629',
              '1665631',
              '1665622',
              '1665623',
              '1665628',
              '1665610'],
             'DK110-DK010 TR102': ['1926392', '1662926', '1926393'],
             'RTW150-RTW025 TR3': ['28961', '29253', '260906'],
             'DDM150-DDM050 TR3': ['1686000', '1685956', '1685997'],
             'DTO-RSD150 W': ['1170902',
              '1170923',
              '1170700',
              '1170717',
              '1170953',
              '1168953',
              '1170770',
              '1170875',
              '1170652

In [ ]:

def flip(conn):
    conn['geometry']['coordinates'] = conn['geometry']['coordinates'][::-1]
    conn['properties']['from_id'], conn['properties']['to_id'] = conn['properties']['to_id'], conn['properties']['from_id']

seen = set()
for connid, c in id2conn.items():
# for connid, c in [('1808383', id2conn['1808383'])]:
    if isconn(c['properties']['from_id']) and isconn(c['properties']['to_id']):
        continue
    if connid in seen:
        continue
    else:
        seen.add(connid)
    if isconn(c['properties']['from_id']) and not isconn(c['properties']['to_id']):
        flip(c)
    prev = connid
    while isconn(c['properties']['to_id']) and not isconn(c['properties']['from_id']) and c['properties']['to_id'] not in seen:
        
        dest = c['properties']['to_id']
        # print(dest)
        
        destconn = id2conn[dest]
        seen.add(dest)
        if destconn['properties']['to_id'] == prev: flip(destconn) # flip if necessary
        try:
            assert destconn['properties']['from_id'] == prev # now start should be the current connection
        except:
            print('Bijlmer Zuid situation, where 3 lines become 1. from and to don"t match\n')
            print(connid, dest, prev)
            print(c['properties'])
            print(destconn['properties'])

        c['geometry']['coordinates'] += destconn['geometry']['coordinates'] # add the coordinates
        c['properties']['connections'] |= destconn['properties']['connections'] # add the connections
        c['properties']['to_id'] = destconn['properties']['to_id'] # update the new destination
        prev = dest
          

Bijlmer Zuid situation, where 3 lines become 1. from and to don"t match

1893443 30649 1893443
{'SE_FLD0_MHSKEY': '1893443|50019', 'SE_FLD1_BEDRIJFSSTATUS': 'In bedrijf', 'SE_FLD4_BOUWJAAR': 126230400000, 'ESRI_OID': 209, 'SE_FLD27_NETSCHAKELID': 'GT150-GT010 TR2', 'SE_FLD32_OBJECTID': '1893443', 'SE_FLD34_OBJECTTYPE': 'HSkabeldeel', 'SE_FLD38_SPANNINGSNIVEAU': 150, 'Shape__Length': 3.135050686061963, 'ID': '1893443', 'Spanning': 150, 'Netschakel': 'GT150-GT010 TR2', 'from_id': False, 'to_id': '30649', 'connections': {'30649', '1893445', '1893444'}}
{'SE_FLD0_MHSKEY': '30649|50019', 'SE_FLD1_BEDRIJFSSTATUS': 'In bedrijf', 'SE_FLD4_BOUWJAAR': 126230400000, 'ESRI_OID': 1769, 'SE_FLD27_NETSCHAKELID': 'GT150-GT010 TR2', 'SE_FLD32_OBJECTID': '30649', 'SE_FLD34_OBJECTTYPE': 'HSkabeldeel', 'SE_FLD38_SPANNINGSNIVEAU': 150, 'Shape__Length': 223.5556751862227, 'ID': '30649', 'Spanning': 150, 'Netschakel': 'GT150-GT010 TR2', 'from_id': 'Geertruidenberg', 'to_id': '1893445', 'connections': {'18934

### Count the stations how many connections they have

In [ ]:
if cluster_stations:
    from collections import defaultdict
    conn = defaultdict(int)
    for v in verbindingen['features']:
        conn[v['properties']['from_id']] += 1
        conn[v['properties']['to_id']] += 1
    conn

In [ ]:
priority = ['Eemshaven', 'Borssele']
station2spanning = {s['properties']['Naam']: s['properties']['Spanning'] for s in stations['features']}

if cluster_stations:
    from collections import defaultdict
    conn = defaultdict(int)
    for v in verbindingen['features']:
        conn[v['properties']['from_id']] += 1
        conn[v['properties']['to_id']] += 1
    conn


    ufstation = UnionFind([])
    for idx, s1 in enumerate(stations['features']):
        for idx2, s2 in enumerate(stations['features'][idx+1:], start=idx+1):
            if use_tennet_stations:
                if dist(s1['geometry']['coordinates'][0][0], s2['geometry']['coordinates'][0][0]) < max_distance:
                    ufstation.union(s1['properties']['Naam'],s2['properties']['Naam'])
            else:
                if dist(s1['geometry']['coordinates'], s2['geometry']['coordinates']) < max_distance:
                    ufstation.union(s1['properties']['Naam'],s2['properties']['Naam'])
    station2mainstation = {}
    removed = set()

    for g in ufstation.groups():
        sortedgroup = sorted(g, key=lambda x: (x not in priority, -station2spanning[x], -conn[x]))
        print(sortedgroup)
        for s in sortedgroup[1:]:
            station2mainstation[s] = sortedgroup[0]
            removed.add(s)
        
else:
    removed = set()
    station2mainstation = {}

['Borssele', 'Zeeuwse Kust Landstation']
['Hengelo Weideweg']
['Vierverlaten']
['Lelystad']
['Diemer Vijfhoek', 'Diemen']
['Ens']
['Krimpen a/d IJssel']
['Eemshaven', 'Eemshaven Oudeschip', 'Eemshaven Converterstation 380 kV', 'Eemshaven Synergieweg 380kV', 'Waddenweg Converterstation 380 kV', 'Eemshaven Comp. en Filteren', 'Robbenplaat', 'Oostpolder', 'Eemshaven Oost']
['Groningen Hunze', 'Groningen Bornholmstraat']
['Bergum']
['Louwsmeer']
['Doetinchem', 'Langerak']
['Wijk aan Zee', 'Hollandse Kust Noord Landstation']
['Rilland']
['TheemswegÂ\xa0', 'Merseyweg']
['Zeyerveen']
['Maasbracht']
['Meeden']
['Westerlee', 'De Lier']
['Bleiswijk', 'Hoge snelheidslijn']
['Geervliet Noorddijk', 'Botlek', 'Geervliet', 'Exxon Mobile Rotterdam']
['Zwolle', 'Hessenweg', 'Zwolle Hessenweg']
['Borculo', 'Borculo Berkel']
['Beersdal', 'Huskensweg']
['Slochteren Dellerweerden', 'Slochteren Overschild']
['Hengelo Boldershoek', 'Hengelo AVI']
['Dodewaard']
['Eerbeek', 'Eerbeek Schoonmansmolen']
['Boxmeer

### Make data for loom

In [ ]:
%pip install pyproj

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
verbindingen['features'] = sorted(verbindingen['features'], key = lambda x: -x['properties']['Spanning'])
scale = 6 # the amount to scale with (higher --> more scaling down)

forbidden = ['Enecogen']
def normalize(coor, scale = scale):
    dx = coor[0] - x
    dy = coor[1] - y
    
    return [x + dx/scale, y + dy/scale]
   
from pyproj import Transformer

# Example: WGS84 (lon/lat) → UTM zone 31N (meters)
proj_to_meters = Transformer.from_crs("EPSG:4326", "EPSG:32631", always_xy=True)
proj_to_lonlat = Transformer.from_crs("EPSG:32631", "EPSG:4326", always_xy=True)

# Original normalize function in lon/lat
def normalize(coor, scale=scale):
    # 1. Convert lon/lat to meters
    mx, my = proj_to_meters.transform(coor[0], coor[1])
    mx0, my0 = proj_to_meters.transform(x, y)
    
    # 2. Do your scaling in meters
    dx = mx - mx0
    dy = my - my0
    mx_new = mx0 + dx / scale
    my_new = my0 + dy / scale
    
    # 3. Convert back to lon/lat
    lon_new, lat_new = proj_to_lonlat.transform(mx_new, my_new)
    return [lon_new, lat_new]
 
    
accepted = {f['properties']['Naam'] for f in stations['features'] if f['properties']['Naam'] not in forbidden} 

loom_nodes = {
    "type": "FeatureCollection",
    "features": []
}

for node in stations['features']:
    if node['properties']['Naam'] in accepted:
        loom_node = {
            "type": "Feature",
            "geometry": node['geometry'].copy(),
            "properties": {
                "id": node['properties']['Naam'],
                "station_label": ' ' + node['properties']['Naam'] + '  '
            }
        }
        # for d in ['deg', 'deg_in', 'deg_out']:
        #     loom_node['properties'][d] =G.degree[node['properties']['Naam']]
        if use_tennet_stations:
            loom_node["geometry"]["coordinates"] = normalize(loom_node["geometry"]["coordinates"][0][0])
            loom_node['geometry']['type'] = 'Point'
        else:
            loom_node["geometry"]["coordinates"] = normalize(loom_node["geometry"]["coordinates"])
            
        loom_nodes['features'].append(loom_node)

# Save to JSON
with open("loom_nodes.json", "w") as f:
    json.dump(loom_nodes, f, indent=2)

print("Conversion done! Saved as loom_nodes.json")

# Convert edges to Loom format
loom_edges = {
    "type": "FeatureCollection",
    "features": []
}

def sample(lst, size=10):
    newlist = lst[::size]
    if newlist[-1] != lst[-1]:
        newlist.append(lst[-1])
    return newlist
    
seen = set()
for edge in verbindingen['features']:
    if 'from_id' in edge['properties'] and edge['properties']['to_id'] and edge['properties']['Spanning'] > spanningscutoff:
        
        spanning = str(round(edge['properties']['Spanning']))
        loom_edge = {
            "type": "Feature",
            "geometry": edge['geometry'].copy(),
            "properties": {
                # "id": edge['properties']['ID'],
                "from": station2mainstation.get(edge['properties']['from_id'], edge['properties']['from_id']),
                "to": station2mainstation.get(edge['properties']['to_id'], edge['properties']['to_id']),
                "dbg_lines": str(round(edge['properties']['Spanning'])),
                "spanning": spanning,
                "lines": [{
                "color": spanning_kleuren.get(int(spanning), 'grey'),
                "id": spanning,
                # "id": edge['properties']['ID'],
                # "label": spanning
                }]
                
                # optional: add a label, color, etc.
            }
        }
        
        loom_edge["geometry"]["coordinates"] = [normalize(c) for c in loom_edge["geometry"]["coordinates"]]
        # loom_edge["geometry"]["coordinates"] = loom_edge["geometry"]["coordinates"][:25] + loom_edge["geometry"]["coordinates"][-25:]
        # loom_edge["geometry"]["coordinates"] = [loom_edge["geometry"]["coordinates"][0], loom_edge["geometry"]["coordinates"][-1]]
        
        loom_edge["geometry"]["coordinates"] = sample(loom_edge["geometry"]["coordinates"], 35)

            
        if loom_edge['properties']['from'] in accepted and loom_edge['properties']['to'] in accepted: 
            # print(loom_edge['properties'])
            if loom_edge['properties']['from'] != loom_edge['properties']['to']: # no self loops
                loom_edge["properties"]["fromto"] = tuple(sorted((loom_edge['properties']['from'], loom_edge['properties']['to'], loom_edge['properties']['spanning'])))
                if loom_edge["properties"]["fromto"] not in seen:
                    seen.add(loom_edge["properties"]["fromto"])
                    loom_edges['features'].append(loom_edge)
                    print(loom_edge['properties']['spanning'])
                else:
                    if accept_multiple_lines:
                        for e in loom_edges['features']:
                            if e['properties']['fromto'] == loom_edge['properties']['fromto']:
                                e['properties']['lines'].append(loom_edge['properties']['lines'][0])
                                break
        else:
            pass
        
    else:
        pass
        # print('not found')
# Save to JSON
with open("loom_edges.json", "w") as f:
    json.dump(loom_edges, f, indent=2)

print("Edges converted to Loom format!")

loom_all = {
    "type": "FeatureCollection",
    "features": loom_nodes['features'] + loom_edges['features']
}

# Optionally save to a file
with open("loom_combined.json", "w") as f:
    json.dump(loom_all, f, indent=2)
from pathlib import Path

wsl_path = Path(r"\\wsl.localhost\Ubuntu-24.04\home\jesse\loom\examples\netkaart3.json")

# Write JSON with LF line endings
with wsl_path.open("w", encoding="utf-8", newline="\n") as f:
    json.dump(loom_all, f, indent=2, ensure_ascii=False)
    f.write("\n")  # make sure file ends with a newline
print("Nodes and edges combined into one Loom JSON!")

# cat examples/netkaart3.json | docker run -i loom loom | docker run -i loom octi | docker run -i loom transitmap -l > netkaart-octilinear.svg

Conversion done! Saved as loom_nodes.json
380
380
380
380
380
380
380
380
380
380
380
380
380
380
380
380
380
380
380
380
380
380
380
380
380
380
380
380
380
380
380
380
380
380
220
220
220
220
220
220
220
220
220
220
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
150
15

### Deploy

In [ ]:
spanningscutoff = 200
use_tennet_stations = True
cutoff = 0.00001 # max distance between connections
# max_distance_line_station = 0.0005 # not too large since Borselle becomes ZKL
max_distance_line_station = 0.004 # not too large since Borselle becomes ZKL
max_distance = 0.01 # max distance between stations

accept_multiple_lines = False # for drawing
scale = 10 # the amount to scale with (higher --> more scaling down)
cluster_stations = True
spanning_kleuren = {
    380: 'red',
    150: 'blue',
    220: 'forestgreen',
    110: 'black'
}  

cat examples/netkaart3.json | ./build/topo | ./build/loom | ./build/octi -g 400 --geo-pen 0.5--nd-move-pen 1 | ./build/transitmap -l --line-width 50 --station-label-textsize 300 > go16.svg

SyntaxError: invalid syntax (3646053654.py, line 18)

In [ ]:
from functools import reduce
for n, conn in netschakels.items():
    found = list((i for i in set.union(*[id2conn[c]['properties']['connections'] for c in conn]) if not isconn(i)))
    if len(found) == 2 and n.count('-')==1:
        print('ok assumed')
    else:
        print(n, found)

MDH150-MDH050 TR4 nvt ['Middelharnis']
ok assumed
DK110-DK010 TR102 ['Dokkum']
RTW150-RTW025 TR3 ['Rotterdam Waalhaven']
DDM150-DDM050 TR3 ['Dordrecht Merwedehaven']
ok assumed
APL150-APL050 TR2 Nvt ['Anna Paulowna']
MDH150-MDH050 TR5 nvt ['Middelharnis']
YPB150-YPB25 TR5 ['Ypenburg']
MDH150-MDH050 TR7 nvt ['Middelharnis']
MVL150-MVL066 TR4 ['Maasvlakte']
MDH150-MDH050 TR6 nvt ['Middelharnis']
APL150-APL050 TR1 Nvt ['Anna Paulowna']
GNHK110-GNHK010 TR112 ['Groningen v. Heemskerkstraat']
YPB150-YPB25 TR2 ['Ypenburg']
SNB110-SNB010 TR9001 ['Schoonebeek']
ZBK150-ZBK020 TR202 ['Zuidbroek']
ok assumed
WTR380-WTR150 TR411 ['Wateringen']
ok assumed
ZBK150-ZBK020 TR101 ['Zuidbroek']
ok assumed
ok assumed
RTW150-RTW025 TR2 ['Rotterdam Waalhaven']
RTW150-RTW025 TR1 ['Rotterdam Waalhaven']
ok assumed
RWD-HRV-OHK110 Z ['Rauwerd', 'Heerenveen', 'Oudehaske']
ok assumed
ok assumed
WDO150-HAS20 W ['Westdorpe']
MVL150-MVL025 TR6 ['Maasvlakte']
WDO150-HAS10 W ['Westdorpe']
GLTK110-GLTK020 TR121 ['Gassel

In [ ]:

docker run -it \
  -v "$PWD":/workspace \
  -v /mnt/c/Users/Gebruiker/AppData/Local/Microsoft/Windows/Fonts:/usr/share/fonts/truetype/windows:ro \
  -w /workspace \
  loom-dev bash


apt update
apt install -y fontconfig

mkdir -p build
cd build
cmake ..
make -j"$(nproc)"
cd ..

SyntaxError: invalid syntax (4011303740.py, line 1)

In [ ]:
# after updating source
cd build && make -j$(nproc) transitmap && cd .. && cat examples/netkaart3.json | ./build/loom | ./build/octi -g 400 --nd-move-pen 0.1 | ./build/transitmap -l --line-width 50 --station-label-textsize 300 > go2.svg

In [ ]:
# outside container
cat examples/netkaart3.json | docker run -i loom topo | docker run -i loom loom | docker run -i loom octi -g 400 --nd-move-pen 1 | docker run -i loom transitmap -l --line-width 50 --station-label-textsize 150  > netkaart-octilinear400-17pen.svg

# inside container
cat examples/netkaart3.json | ./build/loom | ./build/octi -g 400 --nd-move-pen 1 | ./build/transitmap -l --line-width 30 --station-label-textsize 150 > netkaart-svg.svg

KeyError: 'Simonshaven'

### Debug

In [ ]:
inspect_netschakel('ULW-ODR-NIWG150 W')

1391687 1391761 2444106
1391677 1391763 2444108
1391675 1391767 2444110
2303940 2364859 1391767
1391689 1391761 2444106
2303952 1391743 2364877
2303937 2364866 1391763
2303946 1391761 2364866
1390709 1391763 2444108
2303949 1391741 2364859
1391765 1391763 2303943
1391767 2303940 1391765
1391763 2303937 1391765
2303943 1391765 2364877
1391741 2303949 1391694
1391743 2303952 1391694
1391761 2303946 1391743
1389851 Utrecht Lage Weide Utrecht Lage Weide
1389811 Utrecht Lage Weide 1390705
1389809 1390705 Utrecht Lage Weide
1389801 Utrecht Lage Weide Utrecht Lage Weide
1389850 Utrecht Lage Weide 1390713
1389849 1390713 Utrecht Lage Weide
1391694 1391743 2444104
2400780 Utrecht Lage Weide Utrecht Lage Weide
2400784 Utrecht Lage Weide Utrecht Lage Weide
1390711 2444106 1389849
1390707 2444108 1389809
1390504 2444108 1389809
1390705 2444110 1389809
2400792 Utrecht Lage Weide Utrecht Lage Weide
2400776 Utrecht Lage Weide Utrecht Lage Weide
2400772 Utrecht Lage Weide Utrecht Lage Weide
1390715 24

In [ ]:
next(s for s in stations['features'] if s['properties']['Naam']=='Maasbracht')

{'type': 'Feature',
 'id': 140,
 'geometry': {'type': 'Polygon',
  'coordinates': [[[5.91596971156853, 51.1492277460921],
    [5.91598356150681, 51.1492343006882],
    [5.91599019284624, 51.1492282173207],
    [5.91599646222687, 51.1492224735862],
    [5.91643325278072, 51.1488547316054],
    [5.91814557996986, 51.1496660965726],
    [5.9197233897939, 51.1504136387172],
    [5.92041242050136, 51.1507401268944],
    [5.92267314466882, 51.148854786175],
    [5.92071693712344, 51.1479276937978],
    [5.9191437812019, 51.1471820921082],
    [5.91905260922065, 51.1472010810054],
    [5.91887573501117, 51.1471178263698],
    [5.91887766262287, 51.1470510475313],
    [5.91799796766283, 51.1466340838123],
    [5.91685166436674, 51.1460969881532],
    [5.91475931790696, 51.1478467592949],
    [5.91524013928179, 51.1480756076985],
    [5.91462857950013, 51.1485929783806],
    [5.91497870527147, 51.1487587015045],
    [5.9158681147599, 51.1491796593677],
    [5.91596971156853, 51.1492277460921]]]

In [ ]:
std::string color = c.geoms[i].from.line->color();
if (color == "FF0000") {       // red
    strokeWidth *= 3;
} else if (color == "228B22") { // green
    strokeWidth *= 2.25;
} else if (color == "0000FF") { // blue
    strokeWidth *= 1.5;
}





      std::stringstream styleOutlineCropped;
      styleOutlineCropped << "fill:none;stroke:#000000";

      styleOutlineCropped << ";stroke-linecap:butt;stroke-width:"
                          << (_cfg->lineWidth + _cfg->outlineWidth) *
                                 _cfg->outputResolution;
      Params paramsOutlineCropped;
      paramsOutlineCropped["style"] = styleOutlineCropped.str();
      paramsOutlineCropped["class"] += " inner-geom-outline";
      paramsOutlineCropped["class"] +=
          " " + getLineClass(c.geoms[i].from.line->id());

std::stringstream styleStr;
styleStr << "fill:none;stroke:#" << color;
styleStr << ";stroke-linecap:round;stroke-opacity:1;stroke-width:" 
         << strokeWidth;
         

In [ ]:
coortonet = defaultdict(list)
for n in netschakels:
    for conn in netschakels[n]:
        c = id2conn[conn]['geometry']['coordinates'][0]
        coortonet[tuple(c)].append((n,conn))
        c = id2conn[conn]['geometry']['coordinates'][-1] # only the extremeties
        coortonet[tuple(c)].append((n,conn))
print(len(coortonet))
coortonet = {k:v for k,v in coortonet.items() if len(v)>1}
print(len(coortonet))
coortonet = {k:v for k,v in coortonet.items() if len(v)>2}
print(len(coortonet))

27689
26417
227


In [ ]:
getconn(100020786)

{'type': 'Feature',
 'id': 130,
 'geometry': {'type': 'LineString',
  'coordinates': [[4.04315968733697, 52.319317164743],
   [4.04295676213952, 52.3189923460249],
   [4.0429361057921, 52.3189542740331],
   [4.04282175162059, 52.318727861047],
   [4.04281374994022, 52.3187098905829],
   [4.04279900098019, 52.318663716747],
   [4.04279220842141, 52.3186168513345],
   [4.0427934476457, 52.318569807562],
   [4.04280270450842, 52.3185231003237],
   [4.04281987691469, 52.3184772426132],
   [4.04284477805279, 52.3184327356734],
   [4.04287713348993, 52.3183900680643],
   [4.04291659027019, 52.3183497058792],
   [4.04296271542206, 52.3183120936269],
   [4.04301500223235, 52.3182776408204],
   [4.04307288037416, 52.3182467265866],
   [4.04313571476773, 52.3182196888674],
   [4.04320281584372, 52.3181968245368],
   [4.04327344999741, 52.3181783832278],
   [4.0433468424933, 52.318164568265],
   [4.04341196528179, 52.3181537550148],
   [4.04348121925442, 52.3181399603605],
   [4.04355143815524, 5

In [ ]:
'100020786'), ('HZL-HZA220 Z', '100021351'), ('HZL-HZA220 Z', '100029525'

In [ ]:
len(getconn(100020786)['geometry']['coordinates']), len(getconn(100021351)['geometry']['coordinates']), len(getconn(100029525)['geometry']['coordinates']), 

(853, 319, 92)

In [ ]:
len({tuple(c) for c in getconn(100021351)['geometry']['coordinates']} - {tuple(c) for c in getconn(100020786)['geometry']['coordinates']})

0

In [ ]:
cnt = defaultdict(int)
for k,v in coortonet.items():
    cnt[len(v)] += 1
    if len(v)==3:
        print(k,v)
cnt

(4.02533746338502, 51.9839642823864) [('HZL-HZA220 Z', '100020786'), ('HZL-HZA220 Z', '100021351'), ('HZL-HZA220 Z', '100029525')]
(4.02535239309393, 51.9839472678342) [('HZL-HZA220 Z', '100020792'), ('HZL-HZA220 Z', '100021349'), ('HZL-HZA220 Z', '100029528')]
(4.02531996615248, 51.9839483844197) [('HZL-HZA220 Z', '100020795'), ('HZL-HZA220 Z', '100021353'), ('HZL-HZA220 Z', '100029531')]
(4.02518484321409, 51.9838785185283) [('HZL-HZA220 W', '100020955'), ('HZL-HZA220 W', '100021359'), ('HZL-HZA220 W', '100029519')]
(4.02516855243993, 51.9838919305697) [('HZL-HZA220 W', '100020952'), ('HZL-HZA220 W', '100021357'), ('HZL-HZA220 W', '100029516')]
(4.02515382892309, 51.9838772243287) [('HZL-HZA220 W', '100020958'), ('HZL-HZA220 W', '100021355'), ('HZL-HZA220 W', '100029522')]
(4.61999131652187, 52.3051619811259) [('BWK-VHZ380 P', '2051816'), ('BWK-VHZ380 P', '2143062'), ('BWK-VHZ380 P', '2143061')]
(4.62026087314087, 52.30535138499) [('BWK-VHZ380 P', '2051807'), ('BWK-VHZ380 P', '214306

defaultdict(int, {3: 220, 4: 7})

In [ ]:
for co, netschakel_lijnidcomb in coortonet.items():
    netschlist = {element[0] for element in netschakel_lijnidcomb}
    if len(netschlist) > 1:
        print(netschlist, netschakel_lijnidcomb, co)

{'DIM-OZN380 Z', 'KIJ-DIM380 Z'} [('DIM-OZN380 Z', '549299'), ('KIJ-DIM380 Z', '548834'), ('KIJ-DIM380 Z', '549271')] (5.00902008686689, 52.2733371865612)
{'DIM-OZN380 Z', 'KIJ-DIM380 Z'} [('DIM-OZN380 Z', '549299'), ('KIJ-DIM380 Z', '547318'), ('KIJ-DIM380 Z', '549314')] (5.00901642555591, 52.2697425451016)


In [ ]:
for n in netschakels:
    if n.count('-')>1: print(n)

ZKL-ZKL380-F01
ZKL-ZKL380-F02
MVL-SMH-CST380 W
MVL-SMH-CST380 Z
MBT-BMR-DOD380 Z


In [ ]:
for co, netschakel_lijnidcomb in coortonet.items():
    netschlist = {element[0] for element in netschakel_lijnidcomb}
    if len(netschakel_lijnidcomb) > 2:
        print(netschlist, netschakel_lijnidcomb, co)

{'HZL-HZA220 Z'} [('HZL-HZA220 Z', '100020786'), ('HZL-HZA220 Z', '100021351'), ('HZL-HZA220 Z', '100029525')] (4.02533746338502, 51.9839642823864)
{'HZL-HZA220 Z'} [('HZL-HZA220 Z', '100020792'), ('HZL-HZA220 Z', '100021349'), ('HZL-HZA220 Z', '100029528')] (4.02535239309393, 51.9839472678342)
{'HZL-HZA220 Z'} [('HZL-HZA220 Z', '100020795'), ('HZL-HZA220 Z', '100021353'), ('HZL-HZA220 Z', '100029531')] (4.02531996615248, 51.9839483844197)
{'HZL-HZA220 W'} [('HZL-HZA220 W', '100020955'), ('HZL-HZA220 W', '100021359'), ('HZL-HZA220 W', '100029519')] (4.02518484321409, 51.9838785185283)
{'HZL-HZA220 W'} [('HZL-HZA220 W', '100020952'), ('HZL-HZA220 W', '100021357'), ('HZL-HZA220 W', '100029516')] (4.02516855243993, 51.9838919305697)
{'HZL-HZA220 W'} [('HZL-HZA220 W', '100020958'), ('HZL-HZA220 W', '100021355'), ('HZL-HZA220 W', '100029522')] (4.02515382892309, 51.9838772243287)
{'BWK-VHZ380 P'} [('BWK-VHZ380 P', '2051816'), ('BWK-VHZ380 P', '2143062'), ('BWK-VHZ380 P', '2143061')] (4.6199

In [ ]:
netschakels['MBT-BMR-DOD380 Z']

['491832',
 '492897',
 '493918',
 '494416',
 '494510',
 '492679',
 '489612',
 '492899',
 '491283',
 '491561',
 '492743',
 '494188',
 '491582',
 '535600',
 '491191',
 '491189',
 '494061',
 '495139',
 '493966',
 '492510',
 '551633',
 '492530',
 '489583',
 '491800',
 '489563',
 '494194',
 '492184',
 '489781',
 '494378',
 '491722',
 '492170',
 '491710',
 '494180',
 '494085',
 '494911',
 '494017',
 '495063',
 '491275',
 '493930',
 '495157',
 '492905',
 '553194',
 '492182',
 '493264',
 '493916',
 '535146',
 '491461',
 '535122',
 '491153',
 '494943',
 '494125',
 '489934',
 '495403',
 '535755',
 '495085',
 '495129',
 '491555',
 '491211',
 '489229',
 '494172',
 '495306',
 '494460',
 '495171',
 '507535',
 '491566',
 '535495',
 '495470',
 '491579',
 '493134',
 '489858',
 '494917',
 '489928',
 '493424',
 '492168',
 '494358',
 '494001',
 '495045',
 '538947',
 '539433',
 '551660',
 '508142',
 '494955',
 '495165',
 '494087',
 '495328',
 '535190',
 '491215',
 '493224',
 '489539',
 '494182',
 '489492',

In [ ]:
forbidden = {
 'AKU1 (Emmtec)',
 'AL Stoom',
 'AMS 13/14',
 'AMS98',
 'APN',
 'Aardgasbuffer Zuidwending',
 'Agriport-A7',
 'Air Liquide',
 'Aldel (Heveskes)',
 'Amer 6',
 'Amer 7',
 'Amer 9',
 'Ampyr',
 'Attero',
 'Attero Moerdijk',
 'Bergumcentrale',
 'Borssele 30',
 'Borssele Scaldia',
 'Byron Jackson Flowserve',
 'Centrale Moerdijk',
 'Claus A',
 'Claus C',
 'Claus C4',
 'DSM-1 Swentibold',
 'DSM-2 Kerensheide',
 'DSM-3 Neerbeek',
 'DSM-4 Oude Postbaan',
 'Delesto DES-I',
 'Delesto DES-II',
 'Delfzijl Farmsum (Golden Raand)',
 'Dintelhaven Betuweroute',
 'EC20',
 'EC3',
 'EC4',
 'EC5',
 'EC6',
 'EC7',
 'ECL Luttelgeest',
 'ECW Wieringermeer',
 'EH10',
 'EH20',
 'EH30',
 'ELSTA GTG-301',
 'EPNL Rijnmond 1',
 'EPNL Rijnmond II',
 'EPZ EB',
 'ESSO Botlek',
 'EdgeConnex',
 'Eemshaven COBRA',
 'Eemshaven Midden',
 'Eemshaven NorNed',
 'Eemshaven filter',
 'Eindhovencentrale',
 'Eneco Lage Weide 6',
 'Eneco MK12',
 'Enexis Dongecentraleweg',
 'GD Groen',
 'Gasunie Bacton-Balgzand',
 'Gasunie CS Scheemda',
 'Gemini',
 'Google Nimble',
 'Graafstroom Betuweroute',
 'Grijpskerk UGS',
 'HKN Landstation',
 'HKZ-Landstation',
 'HVDC BritNed',
 'Hemweg HW9',
 'Hengelo Salinco',
 'Hydro Agri (Schakelaars)',
 'Hydro Agri Sluiskil',
 'Klantstation Allnex',
 'MPP3',
 'MVL Enecogen',
 'MVL Onyx',
 'Meeden DRT',
 'Moerdijk CCGT',
 'Màximacentrale',
 'NAM Delfzijl Schaapbulten',
 'NAM Menterwolde-Spitsbergen',
 'NAM Scheemda De Eeker',
 'NAM Scheemderzwaag',
 'NAM Schoonebeek',
 'NAM Slochteren Kooipolder',
 'NLR',
 'NOP-Agrowind',
 'NUON Velsen24',
 'NUON Velsen25 PER',
 'NXP Philips',
 'Nobian',
 'NoordZeeWind/OWEZ',
 'Norg UGS',
 'NorthC Blackbox',
 'Nuon Magnum',
 'Oostpolder (Saturn)',
 'PerGen',
 'RWE-EC30',
 'RWE-EC31',
 'Reeweg Havenspoorlijn',
 'RoCa Centrale',
 'Shell Moerdijk',
 'Shell SHH1',
 'Shell shunt',
 'Slochteren Dellerweerden',
 'Sloe (Vlissingen Oost)',
 'Sloe I',
 'Sloe II',
 'Stikstoffabriek Zuidbroek',
 'Swentibold',
 'TAQA (Boekelermeer Zuid)',
 'TATA HSV23',
 'TATA HSV26',
 'TATA HSV28',
 'TATA HVS20',
 'Twence AVI',
 'Twence biomassa',
 'Urenco',
 'Vattenfall Centrale Almere',
 'Vattenfall IJmond 01',
 'Vogelweg HV',
 'WKC Helmond 1/2',
 'WP-Zuidwester',
 'Windpark Blauw',
 'Windpark Bouwdokken',
 'Windpark Friesland',
 'Windpark Krammer',
 'Windpark N33',
 'Windpark Zuidlob',
 'Zeeland Refinery',
 'Zoetermeer-HSL',
 'Zonnepark HVC',
 'Zonnepark Midden Groningen',
 'Zonnepark Stadskanaal'}